# Supplementary Figure S6 - Rank-Instability Mechanism in the Yeast GI-PCC Reference Clustering

Using the same noise-robustness replicates as Supplementary Figure S2, this notebook characterizes the cases where a reference cluster's headline annotation is recovered under noise but does not rank first. Panel A reproduces S2's noise-sensitivity result unchanged. Panel B buckets, for all 7 reference clusters, the rank position of the headline term among each replicate's significant terms. A supporting calculation reports, for reference clusters 3 and 4, each cluster's baseline q-value margin between the headline term and its noise-run displacer, from the unperturbed reference clustering only.

## Setup

Enable inline plotting and define the figure export paths.

In [ ]:
# Enable inline plotting for notebooks
%matplotlib inline

In [ ]:
# Figure export paths and utilities (PNG, 300 DPI).
from pathlib import Path
import warnings

import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message=r"Dropped .* annotations after matrix filtering.*",
    category=RuntimeWarning,
)

NB_ID = "supp_fig_6"
PNG_DIR = Path("png") / NB_ID
PNG_DIR.mkdir(parents=True, exist_ok=True)


def save_figure_png(name, *, fig=None, dpi=300, pad_inches=0.02):
    """Save a Matplotlib figure as a high-quality PNG in the notebook export folder."""
    if fig is None:
        fig = plt.gcf()
    output_path = PNG_DIR / f"{name}.png"
    fig.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=pad_inches,
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    print(f"Saved {output_path}")
    return output_path

## Load Inputs

Load GO Biological Process gene sets and the yeast GI-PCC matrix.

In [ ]:
import json

import pandas as pd

DATA_DIR = Path("data/yeast")
GO_BP_PATH = DATA_DIR / "go_bp_name_to_orfs.json"
MATRIX_PATH = DATA_DIR / "gi_pcc_sampled.tsv"

with GO_BP_PATH.open("r", encoding="utf-8") as fh:
    go_bp = json.load(fh)

DF_GI_PCC = pd.read_csv(MATRIX_PATH, sep="\t", index_col=0)

print(f"GO BP terms loaded: {len(go_bp):,}")
print(f"Matrix shape: {DF_GI_PCC.shape[0]:,} x {DF_GI_PCC.shape[1]:,}")
print(f"Row/column labels identical: {DF_GI_PCC.index.equals(DF_GI_PCC.columns)}")

## Run Reference Clustering

Cluster the full GI-PCC matrix and test GO Biological Process enrichment using the fixed reference settings: Ward/Euclidean linkage, distance threshold 16, minimum cluster size 30, minimum overlap 2, global Benjamini-Hochberg FDR, and `qval <= 0.05`. Matches the configuration used throughout this figure package; not asserted here, consistent with supp_fig_2.ipynb.

In [ ]:
import himalayas
from himalayas import Analysis, Annotations, Matrix

print(f"HiMaLAYAS version: {himalayas.__version__}\n")

REFERENCE_CONFIG = dict(
    linkage_method="ward",
    linkage_metric="euclidean",
    linkage_threshold=16,
    optimal_ordering=True,
    min_cluster_size=30,
)
MIN_OVERLAP = 2
QVAL_CUTOFF = 0.05

matrix = Matrix(DF_GI_PCC)
annotations = Annotations(go_bp, matrix)

analysis = (
    Analysis(matrix, annotations)
    .cluster(**REFERENCE_CONFIG)
    .enrich(min_overlap=MIN_OVERLAP)
    .finalize(col_cluster=True)
)
results = analysis.results
results_sig = results.filter(f"qval <= {QVAL_CUTOFF}")

REFERENCE_N_CLUSTERS = len(results.clusters.cluster_sizes)
REFERENCE_SIG_ROWS = len(results_sig.df)

print(f"Clusters: {REFERENCE_N_CLUSTERS}")
print(f"All enriched rows: {len(results.df):,}")
print(f"Significant cluster-term pairs (q<={QVAL_CUTOFF}): {REFERENCE_SIG_ROWS:,}")

## Panel A: Noise Sensitivity

Add seeded Gaussian noise to the GI-PCC matrix and rerun the reference clustering/enrichment settings at each value in `NOISE_FRACS`. This is the same sweep as supp_fig_2.ipynb's Panel A, copied unchanged; the only addition is caching each replicate's full (unfiltered) per-cluster enrichment table for reference clusters 3 and 4 in `replicate_cache`, so Panels B and C can reuse these exact 25 replicates without regenerating noise.

In [ ]:
import numpy as np

NOISE_FRACS = [0.05, 0.10, 0.20, 0.30, 0.50]
NOISE_SEEDS = [0, 1, 2, 3, 4]
MECH_CLUSTERS = (1, 2, 3, 4, 5, 6, 7)  # reference clusters characterized in Panels B and C

# Reference headline GO term per reference cluster, and each cluster's ORF set.
noise_reference_cluster_labels = results_sig.cluster_labels(rank_by="q", label_mode="top_term")
reference_label_to_cluster = results.clusters.label_to_cluster
reference_cluster_orf_sets = {
    int(cid): {orf for orf, orf_cid in reference_label_to_cluster.items() if orf_cid == cid}
    for cid in noise_reference_cluster_labels["cluster"]
}
REF_TERM = dict(
    zip(
        noise_reference_cluster_labels["cluster"].astype(int),
        noise_reference_cluster_labels["label"],
    )
)


def add_symmetric_noise(values, noise_frac, seed):
    """Add seeded Gaussian noise that is symmetric by construction and has zero diagonal.

    This does not assume the base matrix itself is symmetric; it only ensures the
    injected noise term does not introduce additional row/column asymmetry.
    """
    n = values.shape[0]
    off_diag_mask = ~np.eye(n, dtype=bool)
    off_diag_std = values[off_diag_mask].std()
    rng = np.random.default_rng(seed)
    raw_noise = rng.normal(
        loc=0.0,
        scale=noise_frac * off_diag_std,
        size=(n, n),
    )
    symmetric_noise = np.triu(raw_noise, k=1)
    symmetric_noise = symmetric_noise + symmetric_noise.T
    np.fill_diagonal(symmetric_noise, 0.0)
    return values + symmetric_noise


def best_jaccard_match(reference_orf_set, sweep_label_to_cluster):
    """Return the sweep cluster id with greatest Jaccard overlap to reference_orf_set."""
    sweep_cluster_orfs = {}
    for orf, cid in sweep_label_to_cluster.items():
        sweep_cluster_orfs.setdefault(cid, set()).add(orf)

    best_cid, best_jaccard = None, -1.0
    for cid, sweep_orf_set in sweep_cluster_orfs.items():
        union = len(reference_orf_set | sweep_orf_set)
        jaccard = len(reference_orf_set & sweep_orf_set) / union if union else 0.0
        if jaccard > best_jaccard:
            best_cid, best_jaccard = cid, jaccard
    return best_cid


off_diag_mask = ~np.eye(matrix.values.shape[0], dtype=bool)
off_diag_std = matrix.values[off_diag_mask].std()
print(f"Off-diagonal standard deviation: {off_diag_std:.4f}")

noise_rows = []
n_noise_runs = 0
replicate_cache = {}  # (noise_frac, seed, cluster_id) -> full matched-cluster results table
for noise_frac in NOISE_FRACS:
    recovered_by_cluster = {int(cid): 0 for cid in noise_reference_cluster_labels["cluster"]}
    for seed in NOISE_SEEDS:
        noisy_values = add_symmetric_noise(DF_GI_PCC.values, noise_frac, seed)
        noisy_df = pd.DataFrame(noisy_values, index=DF_GI_PCC.index, columns=DF_GI_PCC.columns)
        noisy_matrix = Matrix(noisy_df)
        noisy_annotations = Annotations(go_bp, noisy_matrix)
        noisy_analysis = (
            Analysis(noisy_matrix, noisy_annotations)
            .cluster(**REFERENCE_CONFIG)
            .enrich(min_overlap=MIN_OVERLAP)
            .finalize(col_cluster=True)
        )
        noisy_results = noisy_analysis.results
        noisy_results_sig = noisy_results.filter(f"qval <= {QVAL_CUTOFF}")
        noisy_label_to_cluster = noisy_results.clusters.label_to_cluster

        for _, ref_row in noise_reference_cluster_labels.iterrows():
            ref_term = ref_row["label"]
            cluster_id = int(ref_row["cluster"])
            ref_orf_set = reference_cluster_orf_sets[cluster_id]
            matched_cid = best_jaccard_match(ref_orf_set, noisy_label_to_cluster)
            matched_sig_terms = set(
                noisy_results_sig.df.loc[noisy_results_sig.df["cluster"] == matched_cid, "term"]
            )
            if ref_term in matched_sig_terms:
                recovered_by_cluster[cluster_id] += 1
            # Cache the full (unfiltered) matched-cluster table for Panels B/C, all 7 clusters.
            if cluster_id in MECH_CLUSTERS:
                replicate_cache[(noise_frac, seed, cluster_id)] = noisy_results.df[
                    noisy_results.df["cluster"] == matched_cid
                ].copy()

        n_noise_runs += 1

    for cid, n_recovered in recovered_by_cluster.items():
        noise_rows.append(
            {
                "ref_cluster_id": cid,
                "noise_frac": noise_frac,
                "top_term_retention_rate": n_recovered / len(NOISE_SEEDS),
            }
        )

noise_retained_top_term = pd.DataFrame(noise_rows)

print(f"Noise reruns completed: {n_noise_runs}")
print("\nHeadline-term retention by reference cluster and noise fraction:")
print(noise_retained_top_term.head(5).to_string(index=False))

## Panel B: Rank-Bucket Stability

Reusing the 25 replicates cached in Panel A, find the reference term's rank among its matched cluster's significant terms (ranked by q-value, ascending) for all 7 reference clusters, and bucket each replicate into #1, top-5, top-10, present outside top-10, or absent entirely. Reported both per noise level and pooled across all 25 replicates.

In [ ]:
def rank_bucket(rank):
    """Bucket a 1-indexed significant-term rank, or None if the term is absent."""
    if rank is None:
        return "not_significant"
    if rank == 1:
        return "#1"
    if rank <= 5:
        return "top-5"
    if rank <= 10:
        return "top-10"
    return "outside_top-10"


def rank_bucket_table(cluster_id):
    """Per-replicate rank bucket of the reference term within its matched cluster."""
    ref_term = REF_TERM[cluster_id]
    rows = []
    for noise_frac in NOISE_FRACS:
        for seed in NOISE_SEEDS:
            full = replicate_cache[(noise_frac, seed, cluster_id)]
            sig = full[full["qval"] <= QVAL_CUTOFF].sort_values("qval").reset_index(drop=True)
            terms = list(sig["term"])
            rank = (terms.index(ref_term) + 1) if ref_term in terms else None
            rows.append(
                {"noise_frac": noise_frac, "seed": seed, "rank": rank, "bucket": rank_bucket(rank)}
            )
    return pd.DataFrame(rows)


rank_bucket_dfs = {}
for cluster_id in MECH_CLUSTERS:
    rdf = rank_bucket_table(cluster_id)
    rank_bucket_dfs[cluster_id] = rdf

    per_noise = (
        rdf.groupby("noise_frac")
        .agg(
            top1_of_5=("rank", lambda r: int((r == 1).sum())),
            top5_of_5=("rank", lambda r: int((r <= 5).sum())),
            top10_of_5=("rank", lambda r: int((r <= 10).sum())),
            present_of_5=("rank", lambda r: int(r.notna().sum())),
        )
        .reset_index()
    )

    n_top1 = int((rdf["rank"] == 1).sum())
    n_top5 = int((rdf["rank"] <= 5).sum())
    n_present = int(rdf["rank"].notna().sum())

    print(f"\nCluster {cluster_id} ({REF_TERM[cluster_id]})")
    print("Per noise level (out of 5 replicates):")
    print(per_noise.to_string(index=False))
    print(f"Pooled (of 25): #1={n_top1}, top-5={n_top5}, present={n_present}")

## Anchor Ratio Calculation

Reports, for reference clusters 3 and 4, the headline term's q-value relative to its noise-run displacer's q-value on the single unperturbed reference clustering computed above. This does not depend on the noise replicates cached in Panel A -- it exists to cite an accurate baseline margin in the caption, not to explain Panel B's rank instability.

In [ ]:
DOMINANT_DISPLACER = {3: "mRNA processing", 4: "ribosome biogenesis"}
ANCHOR_CLUSTERS = (3, 4)  # DOMINANT_DISPLACER is only hand-identified for these two

anchor_rows = []
for cluster_id in ANCHOR_CLUSTERS:
    ref_term = REF_TERM[cluster_id]
    disp_term = DOMINANT_DISPLACER[cluster_id]
    cluster_rows = results.df[results.df["cluster"] == cluster_id]
    ref_qval = cluster_rows.loc[cluster_rows["term"] == ref_term, "qval"].iloc[0]
    disp_qval = cluster_rows.loc[cluster_rows["term"] == disp_term, "qval"].iloc[0]
    anchor_rows.append(
        {
            "cluster": cluster_id,
            "reference_term": ref_term,
            "reference_qval": ref_qval,
            "competitor_term": disp_term,
            "competitor_qval": disp_qval,
            "ratio_competitor_over_reference": disp_qval / ref_qval,
        }
    )

anchor_ratio_table = pd.DataFrame(anchor_rows)
print(anchor_ratio_table.to_string(index=False))

## Render Panels

Export Panels A and B as standalone PNGs for manual figure assembly.

In [ ]:
LABEL_COLOR = "black"
FONT = "Helvetica"
plt.rcParams["font.family"] = FONT
WEAKEST_COLOR = "#e6550d"
STRONGEST_COLOR = "#787878"
PANEL_FIGSIZE = (8, 5)
TITLE_FONTSIZE = 22
TITLE_PAD = 22
AXIS_LABEL_FONTSIZE = 17
TICK_LABEL_FONTSIZE = 15
INSET_TEXT_FONTSIZE = 18
LINEWIDTH = 3
MARKER_SIZE = 9
GRID_COLOR = "#cecece"
GRID_LINEWIDTH = 1
GRID_ALPHA = 0.8

In [ ]:
# Panel A draws one trace per reference cluster at the shared series size; only the
# color separates the weakest trace (WEAKEST_COLOR) from the other clusters.

fig_a, ax_a = plt.subplots(figsize=PANEL_FIGSIZE)
fig_a.patch.set_facecolor("white")

noise_pivot = noise_retained_top_term.pivot(
    index="ref_cluster_id", columns="noise_frac", values="top_term_retention_rate"
).sort_index()
noise_x = np.asarray(noise_pivot.columns, dtype=float)
weakest_cluster = int(noise_pivot.min(axis=1).idxmin())
for cid, row in noise_pivot.iterrows():
    is_worst_case = int(cid) == weakest_cluster
    ax_a.plot(
        noise_x,
        row.values,
        marker="o",
        markersize=MARKER_SIZE,
        linewidth=LINEWIDTH,
        color=WEAKEST_COLOR if is_worst_case else STRONGEST_COLOR,
        zorder=3 if is_worst_case else 2,
    )
ax_a.text(
    0.1,
    0.61,
    "1 of 7 clusters: minimum 0.80",
    ha="left",
    va="bottom",
    fontsize=INSET_TEXT_FONTSIZE,
    color=WEAKEST_COLOR,
    fontname=FONT,
)
ax_a.text(
    0.1,
    0.71,
    "6 of 7 clusters: full recovery",
    ha="left",
    va="bottom",
    fontsize=INSET_TEXT_FONTSIZE,
    color=STRONGEST_COLOR,
    fontname=FONT,
)
ax_a.set_xlim(0.035, 0.515)
ax_a.set_ylim(0.0, 1.04)
ax_a.set_xticks(noise_x)
ax_a.set_xticklabels([f"{v:.2f}" for v in noise_x], fontsize=TICK_LABEL_FONTSIZE, fontname=FONT)
ax_a.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax_a.set_yticklabels(
    ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=TICK_LABEL_FONTSIZE, fontname=FONT
)
ax_a.set_xlabel(
    "Noise fraction",
    fontsize=AXIS_LABEL_FONTSIZE,
    color=LABEL_COLOR,
    fontname=FONT,
)
ax_a.set_ylabel(
    "Headline annotation recovery", fontsize=AXIS_LABEL_FONTSIZE, color=LABEL_COLOR, fontname=FONT
)
ax_a.set_title(
    "Noise sensitivity",
    fontsize=TITLE_FONTSIZE,
    color=LABEL_COLOR,
    fontname=FONT,
    loc="left",
    pad=TITLE_PAD,
)
ax_a.grid(axis="y", color=GRID_COLOR, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

fig_a.tight_layout()
save_figure_png("panel_a_noise_robustness", fig=fig_a)
plt.show()

In [ ]:
import textwrap


# Panel B: rendered table of rank-bucket counts per noise level, for all 7 clusters.
# Collapse to three categories -- #1, Top-5 (not #1), Not Significant -- since top-10-not-top-5
# and outside-top-10 are empty for every cluster (verified: 0/25 in each bucket).
def collapsed_rank_bucket_table(cluster_id):
    """Per-noise-level and pooled counts of #1 / top-5 (not #1) / not-significant."""
    rdf = rank_bucket_dfs[cluster_id]
    rows = []
    for noise_frac in NOISE_FRACS:
        sub = rdf[rdf["noise_frac"] == noise_frac]
        rows.append(
            {
                "Noise Fraction": f"{noise_frac:.2f}",
                "#1": int((sub["rank"] == 1).sum()),
                "Top-5 (not #1)": int(((sub["rank"] > 1) & (sub["rank"] <= 5)).sum()),
                "Not Significant": int(sub["rank"].isna().sum()),
            }
        )
    rows.append(
        {
            "Noise Fraction": "Pooled (25)",
            "#1": int((rdf["rank"] == 1).sum()),
            "Top-5 (not #1)": int(((rdf["rank"] > 1) & (rdf["rank"] <= 5)).sum()),
            "Not Significant": int(rdf["rank"].isna().sum()),
        }
    )
    return pd.DataFrame(rows)


collapsed_rank_bucket_tables = {cid: collapsed_rank_bucket_table(cid) for cid in MECH_CLUSTERS}

# Reference (noise-free) q-value per cluster's headline term, from the single unperturbed
# reference clustering computed above -- not the anchor-ratio competitor q-value.
REF_QVAL = {}
for cid in MECH_CLUSTERS:
    cluster_rows = results.df[results.df["cluster"] == cid]
    REF_QVAL[cid] = float(cluster_rows.loc[cluster_rows["term"] == REF_TERM[cid], "qval"].iloc[0])

fig_b, axes_b = plt.subplots(4, 2, figsize=(13, 19))
fig_b.patch.set_facecolor("white")
axes_b_flat = axes_b.flatten()

for ax, cluster_id in zip(axes_b_flat, sorted(MECH_CLUSTERS)):
    ax.axis("off")
    t = collapsed_rank_bucket_tables[cluster_id]
    tab = ax.table(
        cellText=t.astype(str).values.tolist(),
        colLabels=list(t.columns),
        cellLoc="center",
        loc="center",
        colWidths=[0.34, 0.16, 0.30, 0.28],
    )
    tab.auto_set_font_size(False)
    tab.set_fontsize(13)
    tab.scale(1, 1.7)

    n_rows = len(t)
    for (r, c), cell in tab.get_celld().items():
        cell.set_edgecolor("none")
        cell.set_facecolor("white")
        cell.set_text_props(color="black", fontname=FONT)
        if r == 0:
            cell.set_text_props(color="black", fontname=FONT, fontweight="bold")
            cell.visible_edges = "B"
            cell.set_edgecolor("black")
            cell.set_linewidth(1.4)
        elif r == n_rows:
            cell.set_text_props(color="black", fontname=FONT, fontweight="bold")
            cell.visible_edges = "T"
            cell.set_edgecolor("black")
            cell.set_linewidth(1.2)
        else:
            cell.visible_edges = ""

    wrapped_name_lines = textwrap.wrap(REF_TERM[cluster_id], width=50)
    wrapped_name_lines[-1] += rf" ($\it{{q}}$={REF_QVAL[cluster_id]:.2e})"
    wrapped_name = "\n".join(wrapped_name_lines)

    ax.text(
        0.5,
        1.08,
        f"Cluster {cluster_id}",
        transform=ax.transAxes,
        fontsize=17,
        fontname=FONT,
        fontweight="bold",
        ha="center",
        va="top",
    )
    ax.text(
        0.5,
        0.99,
        wrapped_name,
        transform=ax.transAxes,
        fontsize=13,
        fontname=FONT,
        ha="center",
        va="top",
        linespacing=1.35,
    )

for j in range(len(MECH_CLUSTERS), 8):
    axes_b_flat[j].axis("off")

fig_b.tight_layout(rect=[0, 0, 1, 0.7])
fig_b.subplots_adjust(hspace=0, wspace=0.3)
save_figure_png("panel_b_rank_bucket_all7", fig=fig_b)
plt.show()

## Caption / Claim Boundary

**Supplementary Figure S6. Rank-instability mechanism in the yeast GI-PCC reference clustering under noise.** Both panels reuse the same 25 noise replicates (5 noise fractions x 5 seeds) generated once in Panel A; no additional noise is injected in Panel B.

**Panel A (noise sensitivity)** reproduces Supplementary Figure S2's Panel A unchanged: each reference cluster's headline annotation is retested for presence among its noise-matched cluster's significant terms.

**Panel B (rank-bucket stability)** shows that recovery (Panel A's criterion) is not the same as ranking first, across all 7 reference clusters. Clusters 2, 5, and 6 are fully stable (25/25 #1), and cluster 1 is nearly so (24/25 #1, 1/25 top-5). Two clusters show substantial #1-rank instability despite still being highly recoverable: cluster 4 (cytoplasmic translation) ranks #1 in only 8/25 replicates and cluster 7 (cell division) ranks #1 in only 13/25, yet both remain within the top 5 in every replicate where they are significant at all (cluster 4: 24/25 present, with one true absence at `noise_frac=0.05`, seed 1; cluster 7: 25/25 present). Cluster 3 (mRNA splicing) shows a milder version of the same pattern (18/25 #1, 7/25 top-5, 25/25 present). Across all 7 clusters, no replicate ever places a present headline term outside the top 5.

On the unperturbed reference clustering, cluster 3's headline term already leads its noise-run displacer by a wide baseline margin (q-value ratio approximately 31x), while cluster 4's leads by only approximately 2.2x -- a narrower starting margin consistent with cluster 4 being one of the clusters that destabilizes under noise.

This analysis does not simulate or validate a real biological perturbation; it characterizes, on a single unchanged matrix, how much apparent top-rank disagreement clustering noise alone produces, using the same replicates as Panel A.